# Experiment Sandbox

Rapid prototype notebook. Define the method inline, run on a small slice of the test set, inspect outputs.

In [13]:
# ── Config ──────────────────────────────────────────────────────────────────
MODEL = "meta-llama/llama-3.1-8b-instruct"
BACKEND = "openrouter"
N_SHOTS = 3
PROJECTS = [
    "newton-physics/newton", 
    "orientechnologies/orientdb", 
    "google/adk-java", 
    #"h2oai/h2o-3",
    "apache/airflow",
    #"spring-projects/spring-security"
    ]
SAMPLES_PER_PROJECT = 50
LANGUAGES = ["python", "java"]
DATASET_VERSION = "v3"
MAX_TOKENS = 60
TEMPERATURE = 0.0
MAX_CONCURRENCY = 20

In [14]:
import os, asyncio, random, textwrap, subprocess, re
import pandas as pd
from openai import AsyncOpenAI
from tqdm.asyncio import tqdm as atqdm

from mas_code_sum.data import load_projects
from mas_code_sum.retrievers.bm25 import BM25Retriever
from mas_code_sum.evaluator import bleu as _bleu
from mas_code_sum.methods.few_shot_all_context_instruct import FewShotAllContextInstructSummarizer
from mas_code_sum.methods.zero_shot_context_enriched import _get_metadata_index
from mas_code_sum.enrichers.asap.identifier_extractor import extract_identifier_context
from mas_code_sum.enrichers.file_context import _repo_dir, _extract_func_name_from_code
from mas_code_sum.enrichers.file_context_java import _extract_method_name_from_code as _extract_java_method_name

random.seed(42)

client = AsyncOpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1",
    timeout=60.0,
    max_retries=3,
)

In [15]:
# ── Load test samples ────────────────────────────────────────────────────────
all_projects = load_projects(LANGUAGES, split="test", dataset_version=DATASET_VERSION, projects=PROJECTS)

samples = []
for proj in PROJECTS:
    proj_samples = all_projects[proj]
    picked = random.sample(proj_samples, min(SAMPLES_PER_PROJECT, len(proj_samples)))
    samples.extend(picked)

print(f"{len(samples)} samples from {PROJECTS}")

200 samples from ['newton-physics/newton', 'orientechnologies/orientdb', 'google/adk-java', 'apache/airflow']


In [16]:
# ── BM25 retriever (train set) ───────────────────────────────────────────────
retriever = BM25Retriever(n=N_SHOTS, dataset_version=DATASET_VERSION)

In [17]:
# ── Method 1: simple few-shot ICL (inline) ───────────────────────────────────

def build_messages(sample: dict, examples: list[dict]) -> list[dict]:
    code = " ".join(sample["code_tokens"])

    example_blocks = []
    for ex in examples:
        ex_code = " ".join(ex["code_tokens"])
        ex_doc  = " ".join(ex["docstring_tokens"])
        example_blocks.append(f"Code:\n{ex_code}\nSummary: {ex_doc}")

    examples_text = "\n\n".join(example_blocks)

    user_content = textwrap.dedent(f"""\
        You are a code documentation assistant.

        Here are examples of code summaries. Follow the same style, length, and phrasing.

        {examples_text}

        Now write a one-line summary for the following function.
        Output only the summary text — no explanation, no prefix, no quotes, no extra text.

        Code:
        {code}
        Summary:"""
    )
    return [{"role": "user", "content": user_content}]


async def summarize_one(sample: dict) -> tuple[str, str]:
    """Returns (prediction, prompt)."""
    code = " ".join(sample["code_tokens"])
    language = sample["language"]
    project  = sample.get("repo")
    examples = retriever.retrieve(code, language, project=project)
    messages = build_messages(sample, examples)
    prompt = messages[0]["content"]
    resp = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=MAX_TOKENS,
        temperature=TEMPERATURE,
    )
    raw = resp.choices[0].message.content or ""
    lines = [l.strip() for l in raw.split("\n") if l.strip()]
    return (lines[0] if lines else ""), prompt


async def run_all(samples: list[dict]) -> tuple[list[str], list[str]]:
    sem = asyncio.Semaphore(MAX_CONCURRENCY)
    async def _one(s):
        async with sem:
            return await summarize_one(s)
    results = await atqdm.gather(*[_one(s) for s in samples], desc="method1")
    predictions, prompts = zip(*results)
    return list(predictions), list(prompts)

In [18]:
# ── Method 2: few_shot_all_context_instruct ──────────────────────────────────
method2 = FewShotAllContextInstructSummarizer(
    model=MODEL,
    retriever=retriever,
    backend=BACKEND,
)

In [19]:
# ── Method 3: file context + imported-module context ─────────────────────────
# For Python only: resolve the imports in the target file to repo-local paths,
# then pull signatures + first-doc-lines for the imported names that actually
# appear in the target function.  Java is a no-op (returns "").
import ast as _ast
from pathlib import Path as _Path


def _load_repo_file(repo_dir: _Path, filepath: str, sha: str | None = None) -> str | None:
    if sha:
        result = subprocess.run(
            ["git", "show", f"{sha}:{filepath}"],
            cwd=str(repo_dir), capture_output=True,
        )
        return result.stdout.decode("utf-8", errors="replace") if result.returncode == 0 else None
    try:
        return (repo_dir / filepath).read_text(encoding="utf-8", errors="replace")
    except OSError:
        return None


def _resolve_python_imports(repo_dir: _Path, path: str, sha: str | None = None) -> list[tuple[str, list[str]]]:
    """Return (repo-relative filepath, [imported names]) for each repo-local import in path."""
    src = _load_repo_file(repo_dir, path, sha)
    if not src:
        return []
    try:
        tree = _ast.parse(src)
    except SyntaxError:
        return []

    pkg_parts = list(_Path(path).parent.parts)
    results: list[tuple[str, list[str]]] = []

    for node in tree.body:
        if not isinstance(node, _ast.ImportFrom) or not node.module:
            continue
        names = [alias.asname or alias.name for alias in node.names if alias.name != "*"]
        if not names:
            continue
        mod_parts = node.module.split(".")

        if node.level > 0:
            # Relative import: go up `level - 1` directories from the current package
            up = node.level - 1
            base = pkg_parts[:max(0, len(pkg_parts) - up)]
            candidate_parts = base + mod_parts
        else:
            candidate_parts = mod_parts

        if not candidate_parts:
            continue
        candidate = _Path(*candidate_parts)

        # Try module file, then package __init__
        for probe in [str(candidate) + ".py", str(candidate / "__init__.py")]:
            if (repo_dir / probe).exists():
                results.append((probe, names))
                break

    return results


def _extract_relevant_outline(repo_dir: _Path, filepath: str, relevant: set[str], sha: str | None = None) -> str:
    """Signatures + first doc line for functions/classes in `relevant` found in filepath."""
    src = _load_repo_file(repo_dir, filepath, sha)
    if not src:
        return ""
    try:
        tree = _ast.parse(src)
    except SyntaxError:
        return ""

    file_lines = src.splitlines()

    def _first_doc(node) -> str | None:
        doc = _ast.get_docstring(node)
        return doc.split("\n")[0].strip() if doc else None

    blocks: list[str] = []
    for node in _ast.walk(tree):
        if isinstance(node, (_ast.FunctionDef, _ast.AsyncFunctionDef)) and node.name in relevant:
            sig = file_lines[node.lineno - 1].rstrip()
            doc = _first_doc(node)
            blocks.append(f"{sig}\n    \"{doc}\"" if doc else sig)

    for node in tree.body:
        if isinstance(node, _ast.ClassDef) and node.name in relevant:
            sig = file_lines[node.lineno - 1].rstrip()
            doc = _first_doc(node)
            blocks.append(f"{sig}\n    \"{doc}\"" if doc else sig)

    return "\n\n".join(blocks)


def _get_imported_context(
    project: str, path: str, code: str, language: str,
    sha: str | None = None, max_chars: int = 2000,
) -> str:
    """Build context from modules imported by `path`, restricted to names used in `code`."""
    if language != "python":
        return ""
    try:
        repo_dir = _repo_dir(project)
    except Exception:
        return ""

    # code is tokenised (space-joined tokens) — word split is the reliable identifier extractor
    used = set(re.findall(r"[A-Za-z_][A-Za-z_0-9]*", code))

    resolved = _resolve_python_imports(repo_dir, path, sha)

    parts: list[str] = []
    for filepath, imported_names in resolved:
        relevant = set(imported_names) & used
        if not relevant:
            continue
        snippet = _extract_relevant_outline(repo_dir, filepath, relevant, sha)
        if snippet:
            parts.append(f"# From {filepath}\n{snippet}")

    result = "\n\n".join(parts)
    if len(result) > max_chars:
        result = result[:max_chars].rstrip() + "\n... [truncated]"
    return result


def _build_prompt_m3(
    code: str,
    examples: list[dict],
    project: str | None,
    extra: str,
    imported: str,
) -> str:
    about = _get_metadata_index().get(project, {}).get("about") if project else None
    header = "You are a code documentation assistant."
    if project and about:
        header += f" The target repository is {project}: {about}."
    elif project:
        header += f" The target repository is {project}."

    example_blocks = [
        f"Code:\n{' '.join(ex['code_tokens'])}\nSummary: {' '.join(ex['docstring_tokens'])}"
        for ex in examples
    ]
    examples_section = (
        "Here are examples of code summaries. "
        "Study the style, length, and phrasing pattern — your output must follow the same format.\n\n"
        + "\n\n".join(example_blocks)
    )

    target_parts: list[str] = []
    if extra:
        target_parts.append(f"Here is the outline of the functions within the same file:\n{extra}")
    if imported:
        target_parts.append(f"Here are signatures from imported modules used by this function:\n{imported}")
    target_parts += [
        "Now write a one-line summary for the following function. "
        "Output only the summary text with no explanation, no prefix, no quotes, and no extra text.",
        code,
        "Summary:",
    ]
    return "\n\n".join([header, examples_section, "\n\n".join(target_parts)])


async def run_all_3(samples: list[dict]) -> tuple[list[str], list[str]]:
    sem = asyncio.Semaphore(MAX_CONCURRENCY)

    async def _one(s):
        async with sem:
            code     = " ".join(s["code_tokens"])
            language = s["language"]
            project  = s.get("repo")
            path     = s.get("path")
            sha      = s.get("blame_sha")
            examples = retriever.retrieve(code, language, project=project)

            extra = method2._extra_context(
                code, language, project, path=path,
                blame_timestamp=s.get("authored_timestamp"),
                blame_sha=sha,
            ) or ""

            imported = _get_imported_context(project, path, code, language, sha=sha) if project and path else ""

            prompt = _build_prompt_m3(code, examples, project, extra, imported)
            resp = await client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=MAX_TOKENS,
                temperature=TEMPERATURE,
            )
            raw_out = resp.choices[0].message.content or ""
            lines = [l.strip() for l in raw_out.split("\n") if l.strip()]
            return (lines[0] if lines else ""), prompt

    results = await atqdm.gather(*[_one(s) for s in samples], desc="method3")
    preds, prompts = zip(*results)
    return list(preds), list(prompts)

In [20]:
# ── Run ──────────────────────────────────────────────────────────────────────
predictions_1, prompts_1 = await run_all(samples)

sem2 = asyncio.Semaphore(MAX_CONCURRENCY)
async def _run_method2():
    async def _one(s):
        async with sem2:
            code = " ".join(s["code_tokens"])
            messages = method2.build_messages(
                code, s["language"], s.get("repo"), s.get("path"),
                blame_timestamp=s.get("authored_timestamp"),
                blame_sha=s.get("blame_sha"),
            )
            prompt = messages[0]["content"]
            pred = await method2.async_summarize(
                code, s["language"],
                project=s.get("repo"),
                path=s.get("path"),
                blame_timestamp=s.get("authored_timestamp"),
                blame_sha=s.get("blame_sha"),
            )
            return pred, prompt
    results = await atqdm.gather(*[_one(s) for s in samples], desc="method2")
    preds, prompts = zip(*results)
    return list(preds), list(prompts)

predictions_2, prompts_2 = await _run_method2()
predictions_3, prompts_3 = await run_all_3(samples)

method3: 100%|██████████| 200/200 [00:42<00:00,  4.73it/s]


In [21]:
# ── Per-sample results ───────────────────────────────────────────────────────
rows = []
for s, pred1, prompt1, pred2, prompt2, pred3, prompt3 in zip(
    samples,
    predictions_1, prompts_1,
    predictions_2, prompts_2,
    predictions_3, prompts_3,
):
    ref = " ".join(s["docstring_tokens"])
    rows.append({
        "project":      s.get("repo", ""),
        "language":     s.get("language", ""),
        "reference":    ref,
        "prediction_1": pred1,
        "bleu_1":       round(_bleu([ref], pred1)[0] * 100, 1),
        "prompt_1":     prompt1,
        "prediction_2": pred2,
        "bleu_2":       round(_bleu([ref], pred2)[0] * 100, 1),
        "prompt_2":     prompt2,
        "prediction_3": pred3,
        "bleu_3":       round(_bleu([ref], pred3)[0] * 100, 1),
        "prompt_3":     prompt3,
    })

df = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", 120)
df[["project", "language", "reference",
    "prediction_1", "bleu_1",
    "prediction_2", "bleu_2",
    "prediction_3", "bleu_3"]]

,project,language,reference,prediction_1,bleu_1,prediction_2,bleu_2,prediction_3,bleu_3
0,newton-physics/newton,python,Compute and cache the precomputed - edge set used by SDF - mesh contacts .,Build and optimize collision edges for this mesh.,8.8,Build and store collision edges for this mesh.,8.8,Build and store collision edges for this mesh.,8.8
1,newton-physics/newton,python,Sync io . display_size and io . display_framebuffer_scale with the current pyglet window .,Refresh display metrics based on window and framebuffer sizes.,6.5,Refresh display metrics based on window and framebuffer sizes.,6.5,Refresh display metrics based on window and framebuffer sizes.,6.5
2,newton-physics/newton,python,Return True when a MuJoCo JOINT equality uses quadratic or higher - order terms .,Return whether a polynomial coefficient has a higher-order term.,9.1,Return whether a MuJoCo equality polynomial has a higher-order term.,12.5,Check if a MuJoCo equality polynomial has a higher-order term.,11.8
3,newton-physics/newton,python,Convert RGB channels from linear light to sRGB / display encoding .,Convert a linear texture image to sRGB format.,16.1,Convert a linear texture to sRGB.,15.4,Convert a linear texture to sRGB.,15.4
4,newton-physics/newton,python,Render the articulation selection panel .,Render the selection panel UI.,38.0,"Render the selection panel with options for articulation pattern, joint filters, and link filters.",12.7,"Render the selection panel with options for articulation pattern, joint filters, and link filters.",12.7
...,...,...,...,...,...,...,...,...,...
195,apache/airflow,python,Make an XCom available for tasks to pull .,Store an XCom value for the current task instance.,18.3,Store an XCom value for the task instance.,20.5,Store an XCom value for the task instance.,20.5
196,apache/airflow,python,Returns the task instance specified by task_id for this dag run,Construct a TaskInstance from the database based on the primary key.,8.3,Returns a TaskInstance from the database based on the task ID.,13.9,Returns a TaskInstance for the given task ID.,12.2
197,apache/airflow,python,Go through the dag_runs and update the state based on the task_instance state . Then set DAG runs that are not finis...,Sets unfinished DAG runs to failed state.,3.0,Sets unfinished DAG runs to failed state.,3.0,Sets unfinished DAG runs to failed state.,3.0
198,apache/airflow,python,Gets the MD5 hash of an object in Google Cloud Storage .,Retrieves the MD5 hash of a specified object in a given bucket.,31.1,Retrieves the MD5 hash of an object in Google Cloud Storage.,91.1,Gets the MD5 hash of an object in Google Cloud Storage.,100.0


In [22]:
# ── Summary ──────────────────────────────────────────────────────────────────
cols = ["bleu_1", "bleu_2", "bleu_3"]
summary = df.groupby("project")[cols].mean().round(2)
summary.loc["overall"] = df[cols].mean().round(2)
print(summary.to_string())

                            bleu_1  bleu_2  bleu_3
project                                           
apache/airflow               17.27   22.27   21.06
google/adk-java              30.42   39.17   40.18
newton-physics/newton        16.62   20.48   20.69
orientechnologies/orientdb   17.48   21.26   21.67
overall                      20.45   25.79   25.90


In [24]:
print(df['prompt_3'].values[50])

You are a code documentation assistant. The target repository is orientechnologies/orientdb: OrientDB is the most versatile DBMS supporting Graph, Document, Reactive, Full-Text and Geospatial models in one Multi-Model product. OrientDB can run distributed (Multi-Master), supports SQL, ACID Transactions, Full-Text indexing and Reactive Queries..

Here are examples of code summaries. Study the style, length, and phrasing pattern — your output must follow the same format.

Code:
@ Override public void removeBackgroundExceptionListener ( final OBackgroundExceptionListener listener ) { final List < WeakReference < OBackgroundExceptionListener > > itemsToRemove = new ArrayList <> ( 1 ) ; for ( final WeakReference < OBackgroundExceptionListener > ref : backgroundExceptionListeners ) { final OBackgroundExceptionListener l = ref . get ( ) ; if ( l != null && l . equals ( listener ) ) { itemsToRemove . add ( ref ) ; } } backgroundExceptionListeners . removeAll ( itemsToRemove ) ; }
Summary: Remo